# FlashRank-Pro Training Pipeline
Each cell runs one stage. All output writes directly to Google Drive.
Session breaks are safe — just re-run all, it skips completed stages.

In [ ]:
# SETUP
import os, shutil, subprocess, time

DRIVE = "/content/drive/MyDrive/flashrank-pro"
LOCAL = "/content/flashrank-pro"
os.makedirs(f"{DRIVE}/data", exist_ok=True)
os.makedirs(f"{DRIVE}/models", exist_ok=True)

# Install
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers sentence-transformers accelerate datasets fire tqdm peft openai huggingface-hub torchao>=0.16.0

# Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Get code
if not os.path.exists(LOCAL):
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        import requests, zipfile, io
        resp = requests.get("https://api.github.com/repos/eulogik/flashrank-pro/zipball/main",
                            headers={"Authorization": f"Bearer {token.strip()}", "Accept": "application/vnd.github+json"},
                            timeout=120)
        z = zipfile.ZipFile(io.BytesIO(resp.content))
        _tmp = LOCAL + "-tmp"
        z.extractall(_tmp)
        os.rename(os.path.join(_tmp, os.listdir(_tmp)[0]), LOCAL)
        shutil.rmtree(_tmp, ignore_errors=True)
        print(f"Got code")
    except:
        !git clone https://github.com/eulogik/flashrank-pro.git {LOCAL}

# Restore from Drive
for d in ["data", "models"]:
    src, dst = f"{DRIVE}/{d}", f"{LOCAL}/{d}"
    if os.path.isdir(src) and os.listdir(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Restored {d}/")
    else:
        os.makedirs(dst, exist_ok=True)

# GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.0f}GB)")
else:
    print("No GPU!")

# Status
for name, path in [("data", f"{DRIVE}/data/synthetic_training_data.jsonl"),
                    ("kd", f"{DRIVE}/models/flashrank-pro-base-kd-en"),
                    ("rl", f"{DRIVE}/models/flashrank-pro-base-rl"),
                    ("merged", f"{DRIVE}/models/flashrank-pro-base-merged")]:
    print(f"  {'ok' if os.path.exists(path) else 'pending'}: {name}")

## Stage 1: Generate Training Data (~2.5h)
Writes directly to Drive. Checkpoints survive session breaks.

In [ ]:
os.chdir(LOCAL)
!python training/01_generate_synthetic_data.py \
    --output_path {DRIVE}/data/synthetic_training_data.jsonl \
    --corpus_name sentence-transformers/gooaq \
    --n_queries 50000 \
    --n_negatives 4
if os.path.exists(f"{DRIVE}/data/synthetic_training_data.jsonl"):
    shutil.copy2(f"{DRIVE}/data/synthetic_training_data.jsonl", f"{LOCAL}/data/synthetic_training_data.jsonl")
    print("Copied to local for next stages")

## Stage 2: Knowledge Distillation (~3h on T4)
Trains ModernBERT from teacher soft labels.

In [ ]:
MODEL_SIZE = "base"
os.chdir(LOCAL)
!python training/02_knowledge_distillation.py \
    --model_name answerdotai/ModernBERT-{MODEL_SIZE} \
    --data_path data/synthetic_training_data.jsonl \
    --output_dir {DRIVE}/models/flashrank-pro-{MODEL_SIZE}-kd-en \
    --batch_size 8 \
    --num_epochs 3 \
    --learning_rate 2e-5

## Stage 3: GRPO Reinforcement Learning (~1-2h)
Fine-tunes with RL scoring. Uses LoRA.

In [ ]:
MODEL_SIZE = "base"
os.chdir(LOCAL)
!python training/03_grpo_rl.py \
    --model_path {DRIVE}/models/flashrank-pro-{MODEL_SIZE}-kd-en \
    --data_path data/synthetic_training_data.jsonl \
    --output_dir {DRIVE}/models/flashrank-pro-{MODEL_SIZE}-rl \
    --batch_size 4 \
    --num_epochs 1

## Stage 4: SLERP Merge (~5min)
Merges KD + RL checkpoints.

In [ ]:
import json
MODEL_SIZE = "base"
KD = f"{DRIVE}/models/flashrank-pro-{MODEL_SIZE}-kd-en"
RL = f"{DRIVE}/models/flashrank-pro-{MODEL_SIZE}-rl"
MERGED = f"{DRIVE}/models/flashrank-pro-{MODEL_SIZE}-merged"

os.chdir(LOCAL)
if os.path.exists(MERGED):
    print("Already done")
else:
    available = [p for p in [KD, RL] if os.path.exists(p)]
    if len(available) < 2:
        print(f"Need 2 checkpoints, found {len(available)}")
    else:
        cfg = {"checkpoints": available, "weights": [0.5, 0.5]}
        os.makedirs("configs", exist_ok=True)
        with open("configs/slerp_config.json", "w") as f:
            json.dump(cfg, f)
        !python training/04_slerp_merge.py \
            --config_path configs/slerp_config.json \
            --output_path {MERGED}

## Sanity Check

In [ ]:
os.chdir(LOCAL)
if os.path.exists(MERGED):
    from flashrank_pro import Reranker
    r = Reranker(MERGED, device="cuda" if torch.cuda.is_available() else "cpu")
    results = r.rerank("how to train a neural network",
                       ["Training neural networks requires backpropagation.",
                        "Python is a programming language.",
                        "Gradient descent optimizes loss functions."])
    for i, res in enumerate(results, 1):
        print(f"{i}. [{res['score']:.4f}] {res['text'][:60]}")
else:
    print("No merged model yet")

## Deploy to HuggingFace

In [ ]:
MODEL_SIZE = "base"
MERGED = f"{DRIVE}/models/flashrank-pro-{MODEL_SIZE}-merged"
os.chdir(LOCAL)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN and os.path.exists(MERGED):
        !python scripts/deploy_to_huggingface.py --model_path {MERGED} --repo_id eulogik/flashrank-pro-{MODEL_SIZE}
except:
    print("No HF_TOKEN or no model")